# HE detector diagnosis: why the retrained detectors do not beat he_run5

Three retraining runs (control A, F twice) never met the selection rule (healthy FP <= 305.42 px and tolerant precision >= 62.66 on validation). Before any more training, three questions, rules fixed before the numbers (decision.md 2026-11-09):

**Q1. Is the measurement right, and what is the best F's path could score?** Ground-truth masks fed in as the prediction. Native path (A, he_run5): recall and tolerant precision must be >= 99%, otherwise the scoring is wrong and is fixed first. F's path (truth shrunk to the 1,536 px scale and back, as F's probability map is): reported as F's ceiling, what a perfect model at that size could score; a ceiling below he_run5 (recall 49.19, precision 62.66) would mean F cannot beat he_run5 by construction.

**Q2. Over- or underfitting?** Same measure (threshold 0.35 and each checkpoint's best threshold within the healthy-FP limit) on 150 DDR training photos with haemorrhage (seed 0) vs the 114 DDR validation photos. Rule per checkpoint: training precision more than 10 points above validation (and the bootstrap 95% CI of the difference above 0) = **overfitting**; otherwise, if training precision is also below 62.66 = **low on both (underfitting / loss / labels)**.

**Q3. Where are the errors?** On validation: share of false-mark pixels from the worst 10% of photos; share of false marks on the exudate grader mask and on the optic disc; recall by haemorrhage size (512-level pixels: 1-10, 11-50, 51-200, >200).

Reuses the retraining notebook's own code (data checks, model loading, inference, scoring) by executing its cells; nothing is re-implemented except the training-set pass.

## Parameters

In [ ]:
import os, json, time
from pathlib import Path
os.environ.setdefault("RETINA_ROOT", str(Path.cwd().parent)); os.environ["RETRAIN_DETS"] = "he"; os.environ["RETRAIN_ARM"] = "F"   # F: sets the 1,536 px factor; A and he_run5 use the native path
CK = Path(os.environ.get("RETRAIN_CKPT_DIR", "/root/retrain_ckpts"))
CHECKPOINTS = {   # name: (path or "prod", arm path?)
    "he_run5": ("prod", False),
    "A_e35": (CK / "he_A_s42" / "epoch035.pth", False),
    "F_e15": (CK / "he_F_s42" / "epoch015.pth", True),
    "F_e55": (CK / "he_F_s42" / "epoch055.pth", True)}
N_TRAIN, TRAIN_SEED, NB_DIAG = 150, 0, 2000
OVERFIT_GAP = 10.0   # points of precision, train minus validation
FP_LIMIT, PREC_NEED = 305.42, 62.66
SIZE_BINS = [0, 10, 50, 200, 10**9]

## 1. The retraining notebook's own setup, data checks, models, scoring and known answers

In [ ]:
import nbformat
RN = nbformat.read(Path(os.environ["RETINA_ROOT"]) / "notebooks" / "detector_retraining.ipynb", 4)
RCODE = [c.source for c in RN.cells if c.cell_type == "code"]
MARKERS = ["ROOT = Path(os.environ.get(\"RETINA_ROOT\"", "def read_gt(p):", "def load_prod(det):", "VCACHE = OUT / \"val_cache\"", "KA, KNOWN_ROWS = {}, []"]
for mk in MARKERS:
    cells = [c for c in RCODE if mk in c]; assert len(cells) == 1, (mk, len(cells))
    t0 = time.time(); exec(cells[0], globals()); print("ran retraining cell:", mk[:40], f"{time.time() - t0:.0f}s", flush=True)
assert ARM == "F" and FUNDUS == 1536 and DETS == ["he"]
DOUT = OUT / "he_diagnosis"; DOUT.mkdir(exist_ok=True)

## 2. Q1: measurement known answer (ground truth as the prediction, both paths)

In [ ]:
def gt_prob(r, arm):
    g = read_gt(r.gt_he).astype(np.float32)
    if not arm: return g
    H, W = g.shape; s = float(INFO.loc[r.name, "s_arm"])
    small = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC if s > 1 else cv2.INTER_AREA)   # as scale_img
    return cv2.resize(small, (W, H), interpolation=cv2.INTER_LINEAR)                                       # as prob_map's way back
rows = []
for r in VALM[VALM.group == "DDR"].itertuples():
    for arm in (False, True):
        m = masks512(gt_prob(r, arm), r.name, "he", thrs=[0.5])[0.5]; a, _ = score_A(m, r.name, "he"); b, _ = score_B(m, r.name, "he")
        assert a == b, (r.name, a, b)
        rows.append(dict(name=r.name, group="DDR", path="F 1536" if arm else "native", thr=0.5, **a))
Q1 = pd.DataFrame(rows)
Q1S = Q1.groupby("path").apply(lambda d: pd.Series(dict(recall_pct=100 * d.found.sum() / d.lesions.sum(), prec_tol_pct=100 * d.tp_tol.sum() / d.pred_px.sum()))).round(2)
display(Q1S); Q1S.to_csv(DOUT / "q1_measurement_known_answer.csv")
Q1_OK = bool((Q1S.loc["native"] >= 99).all()); F_CEIL = Q1S.loc["F 1536"]
print("Q1 native:", "PASS: the scoring gives the truth >= 99%" if Q1_OK else "FAIL: scoring wrong; fix before reading Q2/Q3")
print(f"Q1 F ceiling: recall {F_CEIL.recall_pct}, precision {F_CEIL.prec_tol_pct}", "(below he_run5: F cannot beat it by construction)" if F_CEIL.recall_pct < 49.19 or F_CEIL.prec_tol_pct < 62.66 else "(above he_run5: not a limit)")

## 3. Q2: training photos vs validation, same measure

In [ ]:
cand = man[(man.split == "train") & (man.group == "DDR") & (man.gt_he != "")]
cand = cand[[read_gt(p).any() for p in cand.gt_he]]
TRAINM = cand.sample(N_TRAIN, random_state=TRAIN_SEED).reset_index(drop=True)
print(f"DDR training photos with haemorrhage: {len(cand)}; sampled {len(TRAINM)}"); load_cache(TRAINM)
def run_pass(model, rows, arm):
    out = []
    for r in rows.itertuples():
        M = masks512(prob_map(model, cv2.imread(str(ROOT / r.image)), "he", arm, r.name, arm), r.name, "he")
        for t, m in M.items():
            a, f = score_A(m, r.name, "he"); out.append(dict(name=r.name, group=r.group, thr=t, **a))
    return pd.DataFrame(out)
def load_ck(path):
    c = torch.load(path, map_location=DEVICE, weights_only=False); m = smp.Unet(encoder_name="efficientnet-b2", encoder_weights=None, in_channels=3, classes=1)
    m.load_state_dict(c["state_dict"]); return m.to(DEVICE).eval(), c.get("epoch")
RES = {}
for name, (path, arm) in CHECKPOINTS.items():
    f = DOUT / f"pass_{name}.csv"
    if f.exists(): RES[name] = pd.read_csv(f); print(name, "read from", f.name); continue
    model, ep = (load_prod("he"), 35) if path == "prod" else load_ck(path)
    t0 = time.time(); v = run_pass(model, VALM, arm); v["set"] = "val"; tr = run_pass(model, TRAINM, arm); tr["set"] = "train"
    RES[name] = pd.concat([v, tr]); RES[name].to_csv(f, index=False); del model; torch.cuda.empty_cache()
    print(name, "epoch", ep, f"{time.time() - t0:.0f}s", flush=True)
# known answer inside Q2: he_run5 validation at 0.35 must equal the record
s = summary(RES["he_run5"][RES["he_run5"].set == "val"]).set_index("thr").loc[0.35]
assert abs(s.recall_pct - 49.19) <= 0.3 and abs(s.prec_tol_pct - 62.66) <= 0.3 and abs(s.healthy_fp_px - 305.42) <= 6.1, s
print("he_run5 validation reproduces the record:", s.round(2).to_dict())
def prec_rec(d):
    return 100 * d.tp_tol.sum() / max(d.pred_px.sum(), 1), 100 * d.found.sum() / max(d.lesions.sum(), 1)
def boot_diff(tr, va, rng):
    tn, vn = tr.name.unique(), va.name.unique(); tg, vg = tr.set_index("name"), va.set_index("name"); out = []
    for _ in range(NB_DIAG):
        a = tg.loc[rng.choice(tn, len(tn))]; b = vg.loc[rng.choice(vn, len(vn))]
        out.append(100 * a.tp_tol.sum() / max(a.pred_px.sum(), 1) - 100 * b.tp_tol.sum() / max(b.pred_px.sum(), 1))
    return np.percentile(out, [2.5, 97.5])
rng = np.random.default_rng(0); Q2 = []
for name, d in RES.items():
    sv = summary(d[d.set == "val"]); ok = sv[sv.healthy_fp_px <= FP_LIMIT]
    thr_best = float(ok.loc[ok.prec_tol_pct.idxmax(), "thr"]) if len(ok) else float(sv.loc[sv.prec_tol_pct.idxmax(), "thr"])
    for label, t in (("thr 0.35", 0.35), ("best within FP limit" if len(ok) else "best (no thr within FP limit)", thr_best)):
        tr, va = d[(d.set == "train") & (d.thr == t)], d[(d.set == "val") & (d.thr == t) & (d.group == "DDR")]
        # second path for precision / recall: scipy scoring recomputed from the same masks is in the three-path check below; here the hand formula
        ptr, rtr = prec_rec(tr); pva, rva = prec_rec(va); lo, hi = boot_diff(tr, va, rng)
        assert abs(pva - summary(d[(d.set == "val") & (d.thr == t)]).set_index("thr").loc[t].prec_tol_pct) < 1e-9
        verdict = "overfitting" if (ptr - pva > OVERFIT_GAP and lo > 0) else ("low on both" if ptr < PREC_NEED else "train fine, val gap below rule")
        Q2.append(dict(checkpoint=name, setting=label, thr=t, train_prec=ptr, val_prec=pva, gap=ptr - pva, gap_ci_lo=lo, gap_ci_hi=hi, train_recall=rtr, val_recall=rva,
                       val_healthy_fp=sv.set_index("thr").loc[t].healthy_fp_px, verdict=verdict))
Q2 = pd.DataFrame(Q2); display(Q2.round(1)); Q2.to_csv(DOUT / "q2_train_vs_val.csv", index=False)

## 4. Three-path check on the training pass (cv2 / scipy / skimage scoring must agree)

In [ ]:
name = "F_e15"; path, arm = CHECKPOINTS[name]; model, _ = load_ck(path); t = float(Q2[(Q2.checkpoint == name)].thr.iloc[-1]); n_chk = 0
for r in TRAINM.head(30).itertuples():
    m = masks512(prob_map(model, cv2.imread(str(ROOT / r.image)), "he", arm, r.name, arm), r.name, "he", thrs=[t])[t]
    a, _ = score_A(m, r.name, "he"); b, _ = score_B(m, r.name, "he"); c, _ = score_C(m, r.name, "he")
    assert a == b == c, (r.name, a, b, c); stored = RES[name][(RES[name].set == "train") & (RES[name].name == r.name) & (RES[name].thr == t)].iloc[0]
    assert all(int(stored[k]) == a[k] for k in a), (r.name, a); n_chk += 1
del model; torch.cuda.empty_cache(); print(f"three scoring paths agree and match the stored pass on {n_chk} training photos")

## 5. Q3: where the errors are (validation)

In [ ]:
def breakdown(name, t):
    d = RES[name]; va = d[(d.set == "val") & (d.thr == t)]; dd, hh = va[va.group == "DDR"], va[va.group == "healthy"]
    fp_sorted = np.sort(dd.fp_px.values)[::-1]; top = max(1, int(round(0.1 * len(fp_sorted))))
    return dict(checkpoint=name, thr=t, ddr_fp_px_total=int(dd.fp_px.sum()), share_from_worst_10pct_photos=100 * fp_sorted[:top].sum() / max(fp_sorted.sum(), 1),
                ddr_photos_with_any_fp=int((dd.fp_px > 0).sum()), healthy_fp_share_worst_10pct=100 * np.sort(hh.fp_px.values)[::-1][:max(1, int(round(0.1 * len(hh))))].sum() / max(hh.fp_px.sum(), 1))
def where_and_size(name, t):
    path, arm = CHECKPOINTS[name]; model = load_prod("he") if path == "prod" else load_ck(path)[0]
    on_ex = on_disc = fp_tot = 0; sizes = []
    for r in VALM[VALM.group == "DDR"].itertuples():
        m = masks512(prob_map(model, cv2.imread(str(ROOT / r.image)), "he", arm, r.name, arm), r.name, "he", thrs=[t])[t]
        gd = cv2.dilate(GT512[(r.name, "he")].astype(np.uint8), K_TOL) > 0; fp = m & ~gd
        fp_tot += fp.sum(); on_ex += (fp & (cv2.dilate(GT512[(r.name, "ex")].astype(np.uint8), K_TOL) > 0)).sum(); on_disc += (fp & to512(DISC[r.name])).sum()
        k, lab = gt_lab(r.name, "he"); _, found = score_A(m, r.name, "he"); area = np.bincount(lab.ravel(), minlength=k)[1:]
        sizes += list(zip(area, found))
    del model; torch.cuda.empty_cache(); s = pd.DataFrame(sizes, columns=["area", "found"]); s["bin"] = pd.cut(s.area, SIZE_BINS, labels=["1-10", "11-50", "51-200", ">200"])
    rec = s.groupby("bin", observed=False).found.agg(["sum", "count"]); rec["recall_pct"] = 100 * rec["sum"] / rec["count"].clip(lower=1)
    return dict(checkpoint=name, thr=t, fp_on_exudate_pct=100 * on_ex / max(fp_tot, 1), fp_on_disc_pct=100 * on_disc / max(fp_tot, 1)), rec.assign(checkpoint=name, thr=t)
Q3A, Q3B, Q3C = [], [], []
for name in CHECKPOINTS:
    t = float(Q2[Q2.checkpoint == name].thr.iloc[-1]); Q3A.append(breakdown(name, t)); w, rec = where_and_size(name, t); Q3B.append(w); Q3C.append(rec.reset_index())
Q3A, Q3B, Q3C = pd.DataFrame(Q3A), pd.DataFrame(Q3B), pd.concat(Q3C)
display(Q3A.round(1)); display(Q3B.round(1)); display(Q3C.round(1))
Q3A.to_csv(DOUT / "q3_fp_concentration.csv", index=False); Q3B.to_csv(DOUT / "q3_fp_location.csv", index=False); Q3C.to_csv(DOUT / "q3_recall_by_size.csv", index=False)

## 6. Claims (the only source for messages)

In [ ]:
claims = []
add = lambda kind, text, ev: claims.append(dict(kind=kind, claim=text, evidence=ev))
add("measured", f"Q1 ground truth as prediction: " + "; ".join(f"{p} recall {r.recall_pct:.2f} precision {r.prec_tol_pct:.2f}" for p, r in Q1S.iterrows()) + (" (native PASS)" if Q1_OK else " (native FAIL)"), "Section 2")
for r in Q2.itertuples():
    add("measured", f"Q2 {r.checkpoint} {r.setting} (thr {r.thr}): precision train {r.train_prec:.1f} vs validation {r.val_prec:.1f} (gap {r.gap:.1f}, 95% CI {r.gap_ci_lo:.1f} to {r.gap_ci_hi:.1f}); recall train {r.train_recall:.1f} vs validation {r.val_recall:.1f}; rule verdict: {r.verdict}", "Section 3")
for a, b in zip(Q3A.itertuples(), Q3B.itertuples()):
    add("measured", f"Q3 {a.checkpoint} thr {a.thr}: worst 10% of DDR validation photos hold {a.share_from_worst_10pct_photos:.0f}% of false-mark pixels; {b.fp_on_exudate_pct:.0f}% of false marks on the exudate mask, {b.fp_on_disc_pct:.0f}% on the optic disc", "Section 5")
for (ck, t), g in Q3C.groupby(["checkpoint", "thr"]):
    add("measured", f"Q3 {ck} thr {t} recall by haemorrhage size (512 px): " + ", ".join(f"{r.bin} {r.recall_pct:.0f}% of {int(r['count'])}" for _, r in g.iterrows()), "Section 5")
add("untested", "Why any gap exists (loss weighting, labels, capacity) is not tested here; the verdicts only say which kind of problem it is.", "Design")
CL = pd.DataFrame(claims); CL.to_csv(DOUT / "claims.csv", index=False); pd.set_option("display.max_colwidth", 400); display(CL)